In [ ]:
Nama : Abdul Gofir Samjani
NIM : 202310370311136

In [1]:
%pip install fastexcel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 1.7 MB/s  0:00:02 eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [2]:
DATA_PATH = "../data/raw/online_retail_II.xlsx"

In [3]:
import polars as pl

sheet_2009_2010 = pl.read_excel(
    DATA_PATH,
    sheet_name="Year 2009-2010"
)

sheet_2010_2011 = pl.read_excel(
    DATA_PATH,
    sheet_name="Year 2010-2011"
)

df = pl.concat([sheet_2009_2010, sheet_2010_2011])

df.head()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
i64,str,str,i64,datetime[ms],f64,i64,str
489434,"""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,13085,"""United Kingdom"""
489434,"""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom"""
489434,"""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom"""
489434,"""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085,"""United Kingdom"""
489434,"""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,13085,"""United Kingdom"""


In [4]:
df.shape

(1067371, 8)

In [5]:
df.describe()

statistic,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
str,f64,str,str,f64,str,f64,f64,str
"""count""",1.047871e6,"""1067371""","""1062989""",1.067371e6,"""1067371""",1.067371e6,824364.0,"""1067371"""
"""null_count""",19500.0,"""0""","""4382""",0.0,"""0""",0.0,243007.0,"""0"""
"""mean""",537608.149932,null,null,9.938898,"""2011-01-02 21:13:55.394000""",4.649388,15324.638504,null
"""std""",26662.450447,null,null,172.705794,null,123.553059,1697.46445,null
"""min""",489434.0,"""10002""",""" DOORMAT UNION JACK GUNS AND …",-80995.0,"""2009-12-01 07:45:00""",-53594.36,12346.0,"""Australia"""
"""25%""",515305.0,null,null,1.0,"""2010-07-09 09:46:00""",1.25,13975.0,null
"""50%""",537638.0,null,null,3.0,"""2010-12-07 15:28:00""",2.1,15255.0,null
"""75%""",561006.0,null,null,10.0,"""2011-07-22 10:23:00""",4.15,16797.0,null
"""max""",581587.0,"""m""","""wrongly sold sets""",80995.0,"""2011-12-09 12:50:00""",38970.0,18287.0,"""West Indies"""


In [6]:
df.null_count()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
u32,u32,u32,u32,u32,u32,u32,u32
19500,0,4382,0,0,0,243007,0


In [7]:
df.schema

Schema([('Invoice', Int64),
        ('StockCode', String),
        ('Description', String),
        ('Quantity', Int64),
        ('InvoiceDate', Datetime(time_unit='ms', time_zone=None)),
        ('Price', Float64),
        ('Customer ID', Int64),
        ('Country', String)])

In [8]:
df.select([
    pl.col("InvoiceDate").min().alias("tanggal_awal"),
    pl.col("InvoiceDate").max().alias("tanggal_akhir")
])

tanggal_awal,tanggal_akhir
datetime[ms],datetime[ms]
2009-12-01 07:45:00,2011-12-09 12:50:00


In [9]:
df.select([
    pl.col("Invoice").n_unique().alias("jumlah_invoice"),
    pl.col("Customer ID").n_unique().alias("jumlah_customer"),
    pl.col("Country").n_unique().alias("jumlah_negara"),
])

jumlah_invoice,jumlah_customer,jumlah_negara
u32,u32,u32
45331,5943,43


In [10]:
df.group_by("Country").len().sort("len", descending=True).head(10)

Country,len
str,u32
"""United Kingdom""",981330
"""EIRE""",17866
"""Germany""",17624
"""France""",14330
"""Netherlands""",5140
"""Spain""",3811
"""Switzerland""",3189
"""Belgium""",3123
"""Portugal""",2620


In [11]:
df = df.with_columns(
    (pl.col("Quantity") * pl.col("Price")).alias("TotalPrice")
)

df.head()

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,TotalPrice
i64,str,str,i64,datetime[ms],f64,i64,str,f64
489434,"""85048""","""15CM CHRISTMAS GLASS BALL 20 L…",12,2009-12-01 07:45:00,6.95,13085,"""United Kingdom""",83.4
489434,"""79323P""","""PINK CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom""",81.0
489434,"""79323W""",""" WHITE CHERRY LIGHTS""",12,2009-12-01 07:45:00,6.75,13085,"""United Kingdom""",81.0
489434,"""22041""","""RECORD FRAME 7"" SINGLE SIZE """,48,2009-12-01 07:45:00,2.1,13085,"""United Kingdom""",100.8
489434,"""21232""","""STRAWBERRY CERAMIC TRINKET BOX""",24,2009-12-01 07:45:00,1.25,13085,"""United Kingdom""",30.0


In [12]:
df.group_by("Country").agg(
    pl.col("TotalPrice").sum().alias("total_sales")
).sort("total_sales", descending=True).head(10)

Country,total_sales
str,f64
"""United Kingdom""",1.6383e7
"""EIRE""",615519.55
"""Netherlands""",548524.95
"""Germany""",417988.561
"""France""",328191.8
"""Australia""",167129.07
"""Switzerland""",99728.76
"""Spain""",91859.48
"""Sweden""",87809.42
